
## **Data Preprocessing Exercises: Cleaning, Imputation, Encoding, Scaling & Pipelines**

This notebook covers questions and solutions for common data preprocessing tasks using scikit-learn and pandas.



## Q1. Load the Dataset and Understand its Structure

### Objective:
Load the dataset and understand:
- Number of rows and columns
- Feature names
- Data types
- Target column
- Sample observations

Understanding structure is the FIRST step in preprocessing workflow.


In [183]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/piush-ml/review-sentiment-analysis-dataset/refs/heads/main/data/train.csv")


In [184]:
df.shape

(20000, 13)

In [185]:
df.columns

Index(['id', 'review_date', 'title', 'review_text', 'product_category',
       'user_country', 'verified_purchase', 'price', 'discount', 'rating',
       'helpful_votes', 'total_votes', 'sentiment'],
      dtype='object')

In [186]:
df.dtypes

,0
id,int64
review_date,object
title,object
review_text,object
product_category,object
user_country,object
verified_purchase,bool
price,float64
discount,float64
rating,int64


In [187]:
df.head()

,id,review_date,title,review_text,product_category,user_country,verified_purchase,price,discount,rating,helpful_votes,total_votes,sentiment
0,0,2022-01-31,NaN,VERY BAD QUALITY AND DISAPPOINTING EXPERIENCE....,Home,UK,True,69.44,0.30,2,5,6,negative
1,1,2022-12-30,Bad Purchase,Excellent performance and great value. Rating:...,Books,UK,True,40.69,0.49,3,3,6,neutral
2,2,2021-05-10,Best Buy,Very bad quality and disappointing experience....,Books,Australia,True,40.52,0.38,4,1,5,positive
3,3,2022-07-18,Okay Item,Very bad quality and disappointing experi...,Electronics,Australia,True,41.96,0.25,3,9,16,neutral
4,4,2022-02-04,NaN,THIS PRODUCT IS AMAZING AND WORKS PERFECTLY. R...,Clothing,Canada,False,93.21,0.12,5,6,11,positive



## Q2. Perform Detailed Exploratory Data Analysis (EDA)

### Why?
Before preprocessing:
- Detect missing values
- Identify duplicates
- Understand class distribution
- Inspect numerical statistics


In [188]:
df.isna().sum()

,0
id,0
review_date,0
title,1400
review_text,1400
product_category,0
user_country,1400
verified_purchase,0
price,1400
discount,0
rating,0


In [189]:
(df.isnull().sum() / len(df)) * 100

,0
id,0.0
review_date,0.0
title,7.0
review_text,7.0
product_category,0.0
user_country,7.0
verified_purchase,0.0
price,7.0
discount,0.0
rating,0.0


In [190]:
df.duplicated().sum()

np.int64(0)

In [191]:
df.describe()

,id,price,discount,rating,helpful_votes,total_votes
count,20000.000000,18600.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,9999.500000,54.591945,0.251402,3.459100,5.002100,12.022250
std,5773.647028,46.858794,0.144226,1.281487,2.233908,4.836375
min,0.000000,-208.000000,0.000000,1.000000,0.000000,0.000000
25%,4999.750000,30.507500,0.130000,3.000000,3.000000,8.000000
50%,9999.500000,51.180000,0.250000,4.000000,5.000000,12.000000
75%,14999.250000,71.815000,0.380000,5.000000,6.000000,16.000000
max,19999.000000,754.300000,0.500000,5.000000,20.000000,30.000000



## Q3. Perform Advanced Regex Text Cleaning

### Remove:
- HTML Tags: <.*?>
- URLs: http\S+
- Emails: \S+@\S+
- Digits: \d+
- Special Characters: [^a-zA-Z\s]


In [192]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

text_column = 'review_text'
df["clean_text"] = df[text_column].apply(clean_text)

df[[text_column, "clean_text"]].head()

,review_text,clean_text
0,VERY BAD QUALITY AND DISAPPOINTING EXPERIENCE....,very bad quality and disappointing experience ...
1,Excellent performance and great value. Rating:...,excellent performance and great value rating
2,Very bad quality and disappointing experience....,very bad quality and disappointing experience ...
3,Very bad quality and disappointing experi...,very bad quality and disappointing experience ...
4,THIS PRODUCT IS AMAZING AND WORKS PERFECTLY. R...,this product is amazing and works perfectly ra...



## Q4. Handle Missing Values (Mean & Mode Imputation)

### Numerical: Mean or Mode Imputation

### Categorical: Mode Imputation


In [193]:
from sklearn.impute import SimpleImputer

num_cols = df.select_dtypes(include=np.number).columns
cat_cols = df.select_dtypes(include='object').columns

In [194]:
num_imputer = SimpleImputer(strategy="mean")
df[num_cols] = num_imputer.fit_transform(df[num_cols])

In [195]:
cat_imputer = SimpleImputer(strategy="most_frequent")
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

In [196]:
df.isna().sum()

,0
id,0
review_date,0
title,0
review_text,0
product_category,0
user_country,0
verified_purchase,0
price,0
discount,0
rating,0



## Q5. Apply OneHot Encoding

Transforms categorical variables into binary vectors.


In [197]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = ohe.fit_transform(df[cat_cols])

encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out())
encoded_df.head()

,review_date_2019-01-01,review_date_2019-01-02,review_date_2019-01-03,review_date_2019-01-04,review_date_2019-01-05,review_date_2019-01-06,review_date_2019-01-07,review_date_2019-01-08,review_date_2019-01-09,review_date_2019-01-10,...,clean_text_decent product for the price rating,clean_text_excellent performance and great value rating,clean_text_good but could be better rating,clean_text_highly recommend this product rating,clean_text_i love this product will buy again rating,clean_text_nan,clean_text_not worth the money at all rating,clean_text_terrible build quality broke quickly rating,clean_text_this product is amazing and works perfectly rating,clean_text_very bad quality and disappointing experience rating
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0



## Q6. Apply Label Encoding on Target Variable


In [198]:
from sklearn.preprocessing import LabelEncoder

target_column = 'sentiment'

le = LabelEncoder()
df["target_encoded"] = le.fit_transform(df[target_column])

df[[target_column, "target_encoded"]].head()

,sentiment,target_encoded
0,negative,0
1,neutral,1
2,positive,2
3,neutral,1
4,positive,2



## Q7. Apply Target Encoding

Replace category with mean of target.


In [199]:

target_mean = df.groupby(cat_cols[0])["target_encoded"].mean()
df["target_encoded_category"] = df[cat_cols[0]].map(target_mean)

df[[cat_cols[0], "target_encoded_category"]].head()


,review_date,target_encoded_category
0,2022-01-31,1.235294
1,2022-12-30,1.384615
2,2021-05-10,1.428571
3,2022-07-18,1.384615
4,2022-02-04,1.181818



## Q8. Apply StandardScaler


In [200]:

from sklearn.preprocessing import StandardScaler

scaler_std = StandardScaler()
df_std = scaler_std.fit_transform(df[num_cols])

pd.DataFrame(df_std, columns=num_cols).head()


,id,price,discount,rating,helpful_votes,total_votes
0,-1.731964,0.328586,0.336969,-1.138627,-0.000940,-1.245230
1,-1.731791,-0.307648,1.654379,-0.358265,-0.896254,-1.245230
2,-1.731618,-0.311410,0.891668,0.422098,-1.791569,-1.452002
3,-1.731445,-0.279543,-0.009718,-0.358265,1.789689,0.822486
4,-1.731271,0.854613,-0.911103,1.202461,0.446717,-0.211372



## Q9. Apply MinMaxScaler


In [201]:

from sklearn.preprocessing import MinMaxScaler

scaler_mm = MinMaxScaler()
df_mm = scaler_mm.fit_transform(df[num_cols])

print(pd.DataFrame(df_mm, columns=num_cols).head())


        id     price  discount  rating  helpful_votes  total_votes
0  0.00000  0.288309      0.60    0.25           0.25     0.200000
1  0.00005  0.258433      0.98    0.50           0.15     0.200000
2  0.00010  0.258256      0.76    0.75           0.05     0.166667
3  0.00015  0.259753      0.50    0.50           0.45     0.533333
4  0.00020  0.313010      0.24    1.00           0.30     0.366667



## Q10. Apply MaxAbsScaler


In [202]:

from sklearn.preprocessing import MaxAbsScaler

scaler_ma = MaxAbsScaler()
df_ma = scaler_ma.fit_transform(df[num_cols])

print(pd.DataFrame(df_ma, columns=num_cols).head())


        id     price  discount  rating  helpful_votes  total_votes
0  0.00000  0.092059      0.60     0.4           0.25     0.200000
1  0.00005  0.053944      0.98     0.6           0.15     0.200000
2  0.00010  0.053719      0.76     0.8           0.05     0.166667
3  0.00015  0.055628      0.50     0.6           0.45     0.533333
4  0.00020  0.123572      0.24     1.0           0.30     0.366667



## Q11. Convert Date Column & Extract Time Features

Extract:
- Year
- Month
- Day
- Day of Week
- Is Weekend


In [203]:
date_column = 'review_date'

df[date_column] = pd.to_datetime(df[date_column])

df["year"] = df[date_column].dt.year
df["month"] = df[date_column].dt.month
df["day"] = df[date_column].dt.day
df["day_of_week"] = df[date_column].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5,6]).astype(int)

df[[date_column,"year","month","day","is_weekend"]].head()

,review_date,year,month,day,is_weekend
0,2022-01-31,2022,1,31,0
1,2022-12-30,2022,12,30,0
2,2021-05-10,2021,5,10,0
3,2022-07-18,2022,7,18,0
4,2022-02-04,2022,2,4,0



## Q12. Create Lag Features


In [204]:

df = df.sort_values(date_column)

df["lag_1"] = df[num_cols[0]].shift(1)

df[[num_cols[0], "lag_1"]].head()


,id,lag_1
15742,15742.0,NaN
15309,15309.0,15742.0
4244,4244.0,15309.0
4522,4522.0,4244.0
15465,15465.0,4522.0



## Q13. Create Rolling Mean & Std


In [205]:

df["rolling_mean_3"] = df[num_cols[0]].rolling(3).mean()
df["rolling_std_3"] = df[num_cols[0]].rolling(3).std()

df[[num_cols[0], "rolling_mean_3", "rolling_std_3"]].head()


,id,rolling_mean_3,rolling_std_3
15742,15742.0,NaN,NaN
15309,15309.0,NaN,NaN
4244,4244.0,11765.0,6516.974221
4522,4522.0,8025.0,6309.660292
15465,15465.0,8077.0,6399.705384



## Q14. Feature Selection using VarianceThreshold


In [206]:

from sklearn.feature_selection import VarianceThreshold

numeric_df = df.select_dtypes(include=np.number).fillna(0)

selector = VarianceThreshold(threshold=0.0)
selected_features = selector.fit_transform(numeric_df)

print("Selected Feature Shape:", selected_features.shape)


Selected Feature Shape: (20000, 16)



## Q15. Dimensionality Reduction using PCA


In [207]:

from sklearn.decomposition import PCA

pca = PCA(n_components=2)
reduced = pca.fit_transform(numeric_df)

print("Original Shape:", numeric_df.shape)
print("Reduced Shape:", reduced.shape)


Original Shape: (20000, 16)
Reduced Shape: (20000, 2)



## Q16. Train-Test Split


In [208]:

from sklearn.model_selection import train_test_split

X = numeric_df
y = df["target_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)


Train Shape: (16000, 16)
Test Shape: (4000, 16)


## Q17. Define Numeric and Categorical Preprocessing Pipelines

Create a pipelines for:

1. Numerical Features
   - Median Imputation
   - Standard Scaling

2. Categorical Features
   - Mode Imputation
   - OneHot Encoding

In [209]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [210]:
numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

In [211]:
print("Numeric Features:", numeric_features)
print("Categorical Features:", categorical_features)

Numeric Features: ['id', 'price', 'discount', 'rating', 'helpful_votes', 'total_votes', 'target_encoded', 'target_encoded_category', 'is_weekend', 'lag_1', 'rolling_mean_3', 'rolling_std_3']
Categorical Features: ['title', 'review_text', 'product_category', 'user_country', 'sentiment', 'clean_text']


In [212]:
target_column = "sentiment"
# Remove the encoded target column 'target_encoded' from numeric features
numeric_features = [col for col in numeric_features if col != 'target_encoded']
# Remove the original target column 'sentiment' from categorical features
categorical_features = [col for col in categorical_features if col != target_column]

In [213]:
# Numeric Pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

In [214]:
numeric_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [215]:
# Categorical Pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

In [216]:
numeric_transformer

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

## Q18. Combine Numeric and Categorical Pipelines using ColumnTransformer


Use ColumnTransformer to apply:
- Numeric pipeline → only on numeric features
- Categorical pipeline → only on categorical features


In [217]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['id', 'price', 'discount', 'rating',
                                  'helpful_votes', 'total_votes',
                                  'target_encoded_category', 'is_weekend',
                                  'lag_1', 'rolling_mean_3', 'rolling_std_3']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['title', 'review_text', 'product_category',
                                  'user_country', 'clean_text'])])

## End of Notebook